In [322]:
pip install pandas


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [323]:
import pandas as pd
import numpy as np

In [324]:
df = pd.read_csv("results.csv")
df

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False
...,...,...,...,...,...,...,...,...,...
49500,2026-07-07,Switzerland,Colombia,0.0,0.0,FIFA World Cup,Vancouver,Canada,True
49501,2026-07-09,France,Morocco,NaN,NaN,FIFA World Cup,Foxborough,United States,True
49502,2026-07-10,Spain,Belgium,NaN,NaN,FIFA World Cup,Inglewood,United States,True
49503,2026-07-11,Norway,England,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True


In [325]:
df["date"] = pd.to_datetime(df["date"], format = '%Y-%m-%d')
df.sort_values(by="date")
df = df.dropna(subset = ["home_score", "away_score"])

# Feature #1: Elo Rating Construction

In [326]:
elo_record = {}

In [327]:
def expectedVals(teamA_elo, teamB_elo):
    # Expected score for teamA and teamB
    expA = 1/(1 + 10**((teamB_elo-teamA_elo)/400))
    expB = 1 - expA
    return expA, expB

In [328]:
def actualRating(teamA_score, teamB_score):
    # what to do if there are rows with no scores, maybe just delete the rows that don't have score in them
    if teamA_score > teamB_score:
        return  1, 0
    if teamA_score < teamB_score:
        return 0, 1
    else:
        return 0.5, 0.5



In [329]:
def updatedRating(teamA_elo, teamB_elo, teamA_score, teamB_score, k_factor):
    expA, expB = expectedVals(teamA_elo, teamB_elo)
    actA, actB = actualRating(teamA_score, teamB_score)
    newA = teamA_elo + k_factor * (actA - expA)
    newB = teamB_elo + k_factor * (actB - expB)
    return newA, newB

In [330]:
for index, row in df.iterrows():
    # if the current row teams doesnt have an elo in elo_record intialize it to the value of 1500
    if row["home_team"] not in elo_record:
        elo_record[row["home_team"]] = 1500
    if row["away_team"] not in elo_record:
        elo_record[row["away_team"]] = 1500
    # now add 2 new columns for each row with the current elo_record
    df.loc[index, "home_elo"] = elo_record[row["home_team"]]
    df.loc[index, "away_elo"] = elo_record[row["away_team"]]
    # now update the elo_record given the score of the game
    home_elo = elo_record[row["home_team"]]
    away_elo = elo_record[row["away_team"]]
    home_score = row["home_score"]
    away_score = row["away_score"]
    newHome, newAway = updatedRating(home_elo, away_elo, home_score, away_score, k_factor = 20)
    elo_record[row["home_team"]] = newHome
    elo_record[row["away_team"]] = newAway

   


In [331]:
df

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_elo,away_elo
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,1500.000000,1500.000000
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,1500.000000,1500.000000
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,1490.000000,1510.000000
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,1499.424989,1500.575011
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,1500.541911,1499.458089
...,...,...,...,...,...,...,...,...,...,...,...
49496,2026-07-05,Mexico,England,2.0,3.0,FIFA World Cup,Mexico City,Mexico,False,1853.108960,1908.664762
49497,2026-07-06,Portugal,Spain,0.0,1.0,FIFA World Cup,Dallas,United States,True,1908.173379,1986.256626
49498,2026-07-06,United States,Belgium,1.0,4.0,FIFA World Cup,Seattle,United States,False,1787.758307,1856.886925
49499,2026-07-07,Argentina,Egypt,3.0,2.0,FIFA World Cup,Atlanta,United States,True,2002.293031,1740.171433


In [332]:
# for country in elo_record:
#     print(f"Country: {country} Elo: {elo_record[country]}")

# Feature #2 & #3: Avg Goal and Win Rate

In [333]:
home = df[["date", "home_team", "away_team", "home_score", "away_score"]].copy()
home.columns = ["date", "team", "opponent", "goals_scored", "opponent_score"]
home

,date,team,opponent,goals_scored,opponent_score
0,1872-11-30,Scotland,England,0.0,0.0
1,1873-03-08,England,Scotland,4.0,2.0
2,1874-03-07,Scotland,England,2.0,1.0
3,1875-03-06,England,Scotland,2.0,2.0
4,1876-03-04,Scotland,England,3.0,0.0
...,...,...,...,...,...
49496,2026-07-05,Mexico,England,2.0,3.0
49497,2026-07-06,Portugal,Spain,0.0,1.0
49498,2026-07-06,United States,Belgium,1.0,4.0
49499,2026-07-07,Argentina,Egypt,3.0,2.0


In [334]:
away = df[["date", "home_team", "away_team", "home_score", "away_score"]].copy()
away.columns = ["date", "opponent", "team", "opponent_score", "goals_scored"]

In [335]:
columns = list(away.columns)

idxTeam , idxOpp = columns.index("team"), columns.index("opponent")
idxHomeScore, idxOppScore = columns.index("goals_scored"), columns.index("opponent_score")

columns[idxTeam] , columns[idxOpp] = columns[idxOpp] , columns[idxTeam]
columns[idxHomeScore], columns[idxOppScore] = columns[idxOppScore], columns[idxHomeScore]

away = away[columns]

In [336]:
result = pd.concat([home, away], axis = 0)

result = result.sort_values(by = ["team", "date"])
result = result.reset_index(drop=True)

In [337]:
avg_goal = result.groupby('team')['goals_scored'].rolling(window = 5, min_periods = 5).mean()
avg_goal = avg_goal.groupby(level=0).shift()
avg_goal = avg_goal.reset_index(drop=True)

result["avg_goal_scored"] = avg_goal

In [338]:
# create new column that has the win rate for that game
conditions = [
    (result["goals_scored"] > result["opponent_score"]),
    (result["goals_scored"] < result["opponent_score"]),
    (result["goals_scored"] == result["opponent_score"])
]
choices = [1, 0, 0.5]

In [339]:
match_result = np.select(conditions, choices, default = 0)
result["match_result"] = match_result

win_rate = result.groupby("team")["match_result"].rolling(window = 5).mean()
win_rate = win_rate.groupby(level=0).shift()
win_rate = win_rate.reset_index(drop=True)

result["win_rate"] = win_rate

In [340]:
result.drop(columns = "match_result")
result.loc[result["avg_goal_scored"].isna(), "avg_goal_scored"] = 0
result.loc[result["win_rate"].isna(), "win_rate"] = 0
result

,date,team,opponent,goals_scored,opponent_score,avg_goal_scored,match_result,win_rate
0,2012-09-25,Abkhazia,Artsakh,1.0,1.0,0.0,0.5,0.0
1,2012-10-21,Abkhazia,Artsakh,0.0,3.0,0.0,0.0,0.0
2,2013-09-23,Abkhazia,South Ossetia,3.0,0.0,0.0,1.0,0.0
3,2014-06-01,Abkhazia,Occitania,1.0,1.0,0.0,0.5,0.0
4,2014-06-02,Abkhazia,Sápmi,2.0,1.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...
98997,2017-06-29,Åland Islands,Ynys Môn,2.0,0.0,0.8,1.0,0.5
98998,2023-07-09,Åland Islands,Isle of Wight,0.0,2.0,1.0,0.0,0.7
98999,2023-07-10,Åland Islands,Guernsey,1.0,4.0,1.0,0.0,0.6
99000,2023-07-11,Åland Islands,Western Isles,0.0,2.0,1.0,0.0,0.5


# Feature #4: Adding the Home Advantage Label

In [341]:
df["is_neutral"] = df["neutral"].astype(int)
df

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_elo,away_elo,is_neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,1500.000000,1500.000000,0
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,1500.000000,1500.000000,0
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,1490.000000,1510.000000,0
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,1499.424989,1500.575011,0
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,1500.541911,1499.458089,0
...,...,...,...,...,...,...,...,...,...,...,...,...
49496,2026-07-05,Mexico,England,2.0,3.0,FIFA World Cup,Mexico City,Mexico,False,1853.108960,1908.664762,0
49497,2026-07-06,Portugal,Spain,0.0,1.0,FIFA World Cup,Dallas,United States,True,1908.173379,1986.256626,1
49498,2026-07-06,United States,Belgium,1.0,4.0,FIFA World Cup,Seattle,United States,False,1787.758307,1856.886925,0
49499,2026-07-07,Argentina,Egypt,3.0,2.0,FIFA World Cup,Atlanta,United States,True,2002.293031,1740.171433,1


# Do the Merge Operations

In [342]:
team_features = result[["date", "team", "avg_goal_scored", "win_rate"]].copy()

In [343]:
df = pd.merge(df, team_features, left_on = ["date", "home_team"], right_on = ["date", "team"])
df.rename(columns={'avg_goal_scored': 'home_goals_avg', 'win_rate': 'home_win_rate'}, inplace=True)

df = pd.merge(df, team_features, left_on = ["date", "away_team"], right_on = ["date", "team"])
df.rename(columns={'avg_goal_scored': 'away_goals_avg', 'win_rate': 'away_win_rate'}, inplace=True)

In [344]:
df = df.drop(columns=["team_x", "team_y"])
df

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_elo,away_elo,is_neutral,home_goals_avg,home_win_rate,away_goals_avg,away_win_rate
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,1500.000000,1500.000000,0,0.0,0.0,0.0,0.0
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,1500.000000,1500.000000,0,0.0,0.0,0.0,0.0
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,1490.000000,1510.000000,0,0.0,0.0,0.0,0.0
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,1499.424989,1500.575011,0,0.0,0.0,0.0,0.0
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,1500.541911,1499.458089,0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49838,2026-07-05,Mexico,England,2.0,3.0,FIFA World Cup,Mexico City,Mexico,False,1853.108960,1908.664762,0,2.6,1.0,2.2,0.9
49839,2026-07-06,Portugal,Spain,0.0,1.0,FIFA World Cup,Dallas,United States,True,1908.173379,1986.256626,1,2.0,0.8,2.2,0.9
49840,2026-07-06,United States,Belgium,1.0,4.0,FIFA World Cup,Seattle,United States,False,1787.758307,1856.886925,0,2.2,0.6,2.8,0.8
49841,2026-07-07,Argentina,Egypt,3.0,2.0,FIFA World Cup,Atlanta,United States,True,2002.293031,1740.171433,1,2.8,1.0,1.4,0.5


In [345]:
result[result["team"] == "England"].head(10)

,date,team,opponent,goals_scored,opponent_score,avg_goal_scored,match_result,win_rate
24910,1872-11-30,England,Scotland,0.0,0.0,0.0,0.5,0.0
24911,1873-03-08,England,Scotland,4.0,2.0,0.0,1.0,0.0
24912,1874-03-07,England,Scotland,1.0,2.0,0.0,0.0,0.0
24913,1875-03-06,England,Scotland,2.0,2.0,0.0,0.5,0.0
24914,1876-03-04,England,Scotland,0.0,3.0,0.0,0.0,0.0
24915,1877-03-03,England,Scotland,1.0,3.0,1.4,0.0,0.4
24916,1878-03-02,England,Scotland,2.0,7.0,1.6,0.0,0.3
24917,1879-01-18,England,Wales,2.0,1.0,1.2,1.0,0.1
24918,1879-04-05,England,Scotland,5.0,4.0,1.4,1.0,0.3
24919,1880-03-13,England,Scotland,4.0,5.0,2.0,0.0,0.4


In [346]:
team_features[team_features["team"] == "England"].head(10)

,date,team,avg_goal_scored,win_rate
24910,1872-11-30,England,0.0,0.0
24911,1873-03-08,England,0.0,0.0
24912,1874-03-07,England,0.0,0.0
24913,1875-03-06,England,0.0,0.0
24914,1876-03-04,England,0.0,0.0
24915,1877-03-03,England,1.4,0.4
24916,1878-03-02,England,1.6,0.3
24917,1879-01-18,England,1.2,0.1
24918,1879-04-05,England,1.4,0.3
24919,1880-03-13,England,2.0,0.4
